# Final LOSO Evaluation

Downstream evaluation of the **final** LOSO (leave-one-slide-out) runs -- nuclei
and bins, each x {SVG, Xenium colon panel, SVG+raw-source semantic exclusion} --
see `03_integration/autoencoders/scripts/submit_run_loso_final_{nuclei,bins}.sbatch`
for exactly what each variant is and `data/combined/03_integration/autoencoders/
002um/ablation_studies_summary.txt` for how each choice was decided.

This notebook is written to be **defensive about missing data**: as of when this
was first written, most (unit, variant, slide) combinations hadn't finished
running yet. Every loader below skips silently-but-visibly (prints what's
missing) rather than failing outright, so re-running this notebook as more
folds land just fills in more rows without editing anything.

Sections:
1. Setup + discover what's actually on disk
2. Reconstruct cell/bin ordering + load embeddings (no retraining, no model
   forward pass except where explicitly noted in section 6)
3. Clustering-quality metrics (ARI / silhouette / NMI) against bin-level label
   transfer, bins vs. nuclei and SVG vs. Xenium panel
4. Disease prediction (tumor vs. normal) for every slide, every version
5. Spatial smoothing (`refine_domains`)
6. ARI / silhouette / NMI again, post-smoothing
7. Gene expression prediction (Ridge + MLP probes, adapted from
   `gene_expression_probe.py`)
8. Cluster-type annotation via a canonical marker-gene reference
9. Spatial cluster maps for one slide, before/after smoothing, across
   nuclei/bins x SVG/Xenium panel


## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import anndata as ad
import geopandas as gpd
import scanpy as sc
import squidpy as sq
import torch
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

sys.path.insert(0, "..")
from functions.dual_autoencoder import (
    DualAutoencoder, build_pairing, extract_cell_xy, build_spatial_graph,
    refine_domains, evaluate_disease_signal, get_svg_union_mask, get_gene_mask,
)
from functions import eval_metrics as em
from functions import label_transfer as lt

basedir = Path("../data")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
# ---- what each (unit, variant) combo actually is, and how to invoke it ------
# Mirrors submit_run_loso_final_nuclei.sbatch / submit_run_loso_final_bins.sbatch
# EXACTLY -- if those scripts' VARIANT_FLAGS ever change, update here too (this
# is deliberately not auto-derived from the sbatch files, since parsing bash
# reliably is more fragile than just keeping the two in sync by hand -- there
# are only 6 entries).
SLIDES = ["CRC_P1", "CRC_P11mm", "CRC_P11mm_2_cut", "CRC_P2", "CRC_P3_SMAP",
          "CRC_P4_SMAP", "CRC_P5", "Normal_P3", "Normal_P5"]

UNITS = {
    "nuclei": dict(
        bin_size="002um", dir_cellseg="cell_seg_stardist",
        token_filename="cell_token_embeddings.h5ad",
        run_tag_prefix="",  # nuclei's RUN_TAG has no unit-name component
        cap_tag="cap300000",
        spatial_kwargs=dict(delaunay=True, grid=False, n_neighs=6),
    ),
    "bins": dict(
        bin_size="008um", dir_cellseg="grid_bin",
        token_filename="bin_token_embeddings.h5ad",
        run_tag_prefix="_bins",
        cap_tag="cap610000",
        spatial_kwargs=dict(delaunay=False, grid=True, n_neighs=8),
    ),
}

# RUN_TAG suffix per variant, appended after the shared edrop/gdrop/wd/cap/
# oversample tags every final-run invocation carries regardless of variant
# (see run_loso.py's/_bins.py's own RUN_TAG-construction order -- anchorsim
# comes BEFORE wd in that order, gene-selection tag comes AFTER wd).
VARIANTS = {
    "svg": dict(middle_tag="", gene_panel=None),
    "colon_xenium_panel": dict(middle_tag="_panel_colon_xenium", gene_panel="colon_xenium"),
    "svg_semantic_excl_raw_0.2": dict(middle_tag="_anchorsim0.2_simsrc_raw", gene_panel=None),
}


def run_tag(unit, variant):
    u, v = UNITS[unit], VARIANTS[variant]
    # order confirmed against run_loso.py: ... edrop/gdrop ... [anchorsim+simsrc] ...
    # wd ... [panel|gene_selection] ... cap ... oversample_normal
    anchorsim_part = v["middle_tag"] if variant == "svg_semantic_excl_raw_0.2" else ""
    panel_part = v["middle_tag"] if variant == "colon_xenium_panel" else ""
    return (f"{u['run_tag_prefix']}_edrop0.1_gdrop0.1{anchorsim_part}_wd1e-05"
            f"{panel_part}_{u['cap_tag']}_oversample_normal")


def checkpoint_dir(unit):
    return basedir / "combined/03_integration/autoencoders" / UNITS[unit]["bin_size"] / "loso_token_checkpoints"


def dir_output(unit):
    return basedir / "combined/03_integration/autoencoders" / UNITS[unit]["bin_size"]


for unit in UNITS:
    for variant in VARIANTS:
        print(f"{unit:6s} / {variant:28s} -> RUN_TAG={run_tag(unit, variant)!r}")


In [ ]:
# ---- discover what's actually finished on disk right now ----------------
def discover_available(units=UNITS, variants=VARIANTS, slides=SLIDES):
    rows = []
    for unit in units:
        for variant in variants:
            tag = run_tag(unit, variant)
            for slide in slides:
                npz_path = checkpoint_dir(unit) / f"loso_token_{slide}{tag}_embeddings.npz"
                pt_path = checkpoint_dir(unit) / f"loso_token_{slide}{tag}.pt"
                rows.append({
                    "unit": unit, "variant": variant, "slide": slide,
                    "npz_exists": npz_path.exists(), "pt_exists": pt_path.exists(),
                    "npz_path": npz_path, "pt_path": pt_path,
                })
    return pd.DataFrame(rows)


availability = discover_available()
print(f"{availability['npz_exists'].sum()} / {len(availability)} (unit, variant, slide) embeddings present")
availability.groupby(["unit", "variant"])[["npz_exists", "pt_exists"]].sum()


## 2. Reconstruct cell/bin ordering + load embeddings

`run_leave_one_slide_out_token`'s saved `_embeddings.npz` has `train_z_trans`/
`train_z_token`/`train_slide_name`/`held_out_z_trans`/`held_out_z_token` but
**no spatial coordinates or cell/bin IDs** -- reconstructing them requires
re-running the same deterministic (no randomness, no model forward pass)
data-loading + `build_pairing`/token-restriction steps `run_loso.py` itself
runs before training, exactly as `plot_loso_held_out.py` and
`new_single_omics.ipynb`'s `run_loso_ari_fold` already do. This ONLY works
because those steps are pure key-matching -- the row-count assert below is
the safety check that catches it if that ever stops being true.

In [ ]:
def reconstruct_held_out_cells(unit, held_out_slide):
    """
    Cheap reconstruction (no model forward pass) of one held-out slide's
    cell/bin ordering, coordinates, and independent trans-only cluster labels
    -- mirrors plot_loso_held_out.py exactly, generalized over unit (nuclei
    uses nuclei_geometry.parquet for coords; bins carry obsm['spatial']
    directly, no side-car file -- see extract_cell_xy's own None-safe
    priority order).

    Returns (cell_xy, cluster_labels, trans_sub) or None if this slide has no
    token data for this unit (skipped, matching run_loso.py's/_bins.py's own
    token-availability restriction).
    """
    u = UNITS[unit]
    dir_transcriptomics = basedir / "combined/01_preprocessing" / u["dir_cellseg"] / u["bin_size"] / "custom_binning"

    f_trans_adata = ad.read_h5ad(dir_transcriptomics / "count_area_filtered_adata.h5ad")
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"].astype(str) == held_out_slide].copy()
    trans_global_id = f_trans_adata.obs["Slide_ID"].astype(str) + "_" + f_trans_adata.obs["id"].astype(str)
    f_trans_adata.obs = f_trans_adata.obs.set_index(trans_global_id)

    geom_path = dir_transcriptomics / "nuclei_geometry.parquet"
    f_trans_coord = gpd.read_parquet(geom_path) if geom_path.exists() else None
    if f_trans_coord is not None:
        f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"] == held_out_slide].copy()

    token_path = basedir / held_out_slide / "01_preprocessing/unimodel/metadata" / u["token_filename"]
    if not token_path.exists():
        print(f"  [{unit}] no {u['token_filename']} for {held_out_slide} -- skipping")
        return None
    f_token_adata = sc.read_h5ad(token_path)
    token_key = f_token_adata.obs["Slide_ID"].astype(str) + "_" + f_token_adata.obs["id"].astype(str)
    f_token_adata.obs = f_token_adata.obs.set_index(token_key)
    f_token_adata = f_token_adata[f_token_adata.obs["matched"].astype(bool)].copy()

    if unit == "nuclei":
        dir_histology = basedir / "combined/01_preprocessing/unimodel/metadata"
        f_hist_adata = sc.read_h5ad(dir_histology / "filtered_WSI_patch_embeddings.h5ad")
        f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"] == held_out_slide].copy()
        hist_global_id = f_hist_adata.obs["Slide_ID"].astype(str) + "_" + f_hist_adata.obs["Patch_ID"].astype(str)
        f_hist_adata.obs = f_hist_adata.obs.set_index(hist_global_id)
        keep_mask, _, _ = build_pairing(f_trans_adata, f_hist_adata)
    else:
        # bins have no histology-patch pairing step in the bin-level pipeline
        # (see run_loso_bins.py) -- every bin with token data is kept.
        keep_mask = np.ones(f_trans_adata.n_obs, dtype=bool)

    has_token = f_trans_adata.obs_names.isin(f_token_adata.obs_names)
    keep_mask = keep_mask & has_token
    trans_sub = f_trans_adata[keep_mask].copy()

    cell_xy = extract_cell_xy(trans_sub, f_trans_coord)
    cluster_labels = trans_sub.obs["clusters"].values
    return cell_xy, cluster_labels, trans_sub


def load_embeddings(unit, variant, held_out_slide):
    """Loads a finished fold's saved npz -- returns None (with a printed
    reason) if it doesn't exist yet or the row-count reconstruction fails."""
    tag = run_tag(unit, variant)
    npz_path = checkpoint_dir(unit) / f"loso_token_{held_out_slide}{tag}_embeddings.npz"
    if not npz_path.exists():
        return None

    recon = reconstruct_held_out_cells(unit, held_out_slide)
    if recon is None:
        return None
    cell_xy, cluster_labels, trans_sub = recon

    npz = np.load(npz_path)
    z_trans, z_token = npz["held_out_z_trans"], npz["held_out_z_token"]
    if z_trans.shape[0] != cell_xy.shape[0]:
        print(f"  [{unit}/{variant}/{held_out_slide}] row mismatch: "
              f"{z_trans.shape[0]} embeddings vs {cell_xy.shape[0]} reconstructed cells -- skipping "
              f"(see reconstruct_held_out_cells' docstring before trusting a fix)")
        return None

    return dict(
        cell_xy=cell_xy, cluster_labels=cluster_labels, trans_sub=trans_sub,
        z_trans=z_trans, z_token=z_token,
        z_joint=np.concatenate([z_trans, z_token], axis=1),  # same convention as plot_loso_held_out.py
        train_z_trans=npz["train_z_trans"], train_z_token=npz["train_z_token"],
        train_slide_name=npz["train_slide_name"],
        npz_path=npz_path,
    )


## 3. Clustering-quality metrics (ARI / silhouette / NMI) vs. bin-level label transfer

Reference labels come from `functions/label_transfer.py`'s bin-level output
(`{sample}_grid_predictions.parquet`, columns `predicted_tissuetype_grid_image`
/ `predicted_tissuetype_grid_transcriptomics`) -- **tissue-architecture region
labels** (Neoplasm / Vessel / Connective Tissue / ... from QuPath-style
annotation), not single-cell types (there is no cell-type atlas in this
project yet -- see section 8 for that). For nuclei, each cell's reference
label is the NEAREST 8um-grid cell's label (`assign_nearest_grid_label`,
matching `new_single_omics.ipynb`'s own pattern); for bins, the same function
is reused for consistency even though bins already sit close to the label-
transfer grid's own resolution.

In [ ]:
def load_grid_reference(slide, label_col="predicted_tissuetype_grid_image"):
    """Bin-level label-transfer output for one slide, or None if it hasn't
    been produced (see functions/label_transfer.py's main_grid)."""
    path = basedir / slide / "02_baselines/label_transfer" / f"{slide}_grid_predictions.parquet"
    if not path.exists():
        print(f"  no label-transfer grid predictions for {slide} ({path})")
        return None
    return pd.read_parquet(path, columns=["pxl_col", "pxl_row", label_col])


LEIDEN_RESOLUTION = 0.5  # matches new_single_omics.ipynb's own precedent for this
                          # project -- a knob, not a validated-optimal value; revisit
                          # if a fold's cluster count looks degenerate (1 giant
                          # cluster, or nearly as many clusters as cells)


def cluster_embedding(z, resolution=LEIDEN_RESOLUTION, n_neighbors=15, seed=0):
    """Leiden community detection on a kNN graph built directly in embedding
    space -- matches what thesis Section 5.6.2 actually describes (non-convex,
    graph-local structure; no fixed k, a resolution hyperparameter instead),
    replacing this notebook's earlier KMeans (which needed a fixed k borrowed
    from an external reference clustering -- gone now, Leiden finds its own
    cluster count). Wraps z in a throwaway AnnData purely so sc.pp.neighbors/
    sc.tl.leiden have something to operate on."""
    adata = ad.AnnData(np.asarray(z, dtype=np.float32))
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep="X", random_state=seed)
    # flavor="leidenalg" pinned explicitly: scanpy's default is changing to
    # "igraph" in a future release, which needs the separate python-igraph
    # package -- not confirmed installed in this project's cluster env, so
    # pin the current default rather than silently depending on it.
    sc.tl.leiden(adata, resolution=resolution, random_state=seed, flavor="leidenalg")
    return adata.obs["leiden"].to_numpy()


def evaluate_clustering_quality(unit, variant, held_out_slide, label_col="predicted_tissuetype_grid_image"):
    """One row of ARI/NMI (vs. bin-level label transfer) + silhouette
    (unsupervised) per embedding (z_trans, z_token, z_joint). Returns a list
    of dict rows (one per embedding), or [] if inputs are missing."""
    data = load_embeddings(unit, variant, held_out_slide)
    if data is None:
        return []
    grid_ref = load_grid_reference(held_out_slide, label_col)
    if grid_ref is None:
        return []

    reference = lt.assign_nearest_grid_label(data["cell_xy"], grid_ref, label_col=label_col)

    rows = []
    for emb_name in ["z_trans", "z_token", "z_joint"]:
        z = data[emb_name]
        labels = cluster_embedding(z)
        quality = em.cluster_quality_metrics(z, labels)
        rows.append({
            "unit": unit, "variant": variant, "held_out_slide": held_out_slide, "embedding": emb_name,
            "n_units": z.shape[0], "n_clusters_found": len(np.unique(labels)),
            "ari_vs_label_transfer": em.ari_between_labelings(labels, reference),
            "nmi_vs_label_transfer": em.nmi_between_labelings(labels, reference),
            **quality,
        })
    return rows


clustering_rows = []
for unit in UNITS:
    for variant in VARIANTS:
        for slide in SLIDES:
            clustering_rows.extend(evaluate_clustering_quality(unit, variant, slide))

clustering_quality_df = pd.DataFrame(clustering_rows)
clustering_quality_df.to_csv(basedir / "combined/03_integration/autoencoders/final_loso_clustering_quality.csv", index=False)
clustering_quality_df


In [ ]:
# Summary: bins vs nuclei, svg vs xenium panel -- mean ARI/NMI/silhouette per
# (unit, variant, embedding), collapsed across whichever slides are available
# so far.
if len(clustering_quality_df):
    (clustering_quality_df
     .groupby(["unit", "variant", "embedding"])[["ari_vs_label_transfer", "nmi_vs_label_transfer", "silhouette"]]
     .agg(["mean", "count"]))
else:
    print("Nothing to summarize yet -- no (unit, variant, slide) has both an embedding and a label-transfer reference on disk.")


## 4. Disease prediction for every slide, every version

Reuses `evaluate_disease_signal` directly (same function used throughout the
ablation campaign's disease-signal validation -- see
`ablation_studies_summary.txt`), on the `train_z_trans`/`held_out_z_trans`
already sitting in each fold's saved npz -- no retraining, no model forward
pass needed here.

In [ ]:
disease_rows = []
for unit in UNITS:
    for variant in VARIANTS:
        for slide in SLIDES:
            data = load_embeddings(unit, variant, slide)
            if data is None:
                continue
            held_slide_name = np.full(data["z_trans"].shape[0], slide)
            result = evaluate_disease_signal(
                data["train_z_trans"], data["train_slide_name"], data["z_trans"], held_slide_name
            )
            result.update(unit=unit, variant=variant, held_out_slide=slide)
            disease_rows.append(result)

disease_df = pd.DataFrame(disease_rows)
disease_df.to_csv(basedir / "combined/03_integration/autoencoders/final_loso_disease_signal.csv", index=False)
disease_df


In [ ]:
if len(disease_df):
    tumor = disease_df[disease_df["held_out_true_label"] == 1]
    normal = disease_df[disease_df["held_out_true_label"] == 0]
    print("=== tumor-slide held_out_acc, mean per (unit, variant) ===")
    print(tumor.groupby(["unit", "variant"])["held_out_acc"].mean())
    print()
    print("=== normal-slide held_out_acc, per (unit, variant, slide) -- only 2 normal")
    print("slides total (Normal_P3, Normal_P5), so read individually, not just the mean ===")
    print(normal[["unit", "variant", "held_out_slide", "held_out_acc"]].to_string(index=False))
else:
    print("Nothing yet.")


## 5. Spatial smoothing (`refine_domains`)

Applies both `refine_domains` modes ("hard" majority-vote, "spagcn"
conditional-majority) to the same Leiden clustering used in section 3, one
slide at a time (spatial connectivity must never mix slides -- same caveat
`evaluate_latent_spatial_autocorr`/`token_retrieval_accuracy`'s
`neighbor_tolerant` already document). Each embedding is clustered ONCE per
(unit, variant, slide) and both smoothing modes are applied to that same raw
partition, rather than re-clustering per mode -- avoids redundant Leiden
runs and guarantees "hard" and "spagcn" are being compared starting from an
identical unsmoothed baseline.

In [ ]:
def build_slide_graph(unit, cell_xy):
    """This slide's own spatial connectivity graph (unit-appropriate:
    Delaunay for nuclei, grid/Moore-neighborhood for bins -- see UNITS'
    spatial_kwargs). build_spatial_graph needs an AnnData with obsm['spatial']
    + a library_key; a throwaway single-slide AnnData is the cheapest way to
    reuse it as-is rather than re-deriving the connectivity math here."""
    tmp = ad.AnnData(np.zeros((cell_xy.shape[0], 1)))
    tmp.obsm["spatial"] = cell_xy
    tmp.obs["Slide_ID"] = "single_slide"  # single value -- build_spatial_graph's
                                            # cross-sample-edge assert is trivially satisfied
    return build_spatial_graph(tmp, library_key="Slide_ID", **UNITS[unit]["spatial_kwargs"])


smoothing_examples = {}  # (unit, variant, slide, embedding) -> {raw, hard, spagcn} -- reused in section 6
for unit in UNITS:
    for variant in VARIANTS:
        for slide in SLIDES:
            data = load_embeddings(unit, variant, slide)
            if data is None:
                continue
            conn = build_slide_graph(unit, data["cell_xy"])
            for emb_name in ["z_trans", "z_token", "z_joint"]:
                raw = cluster_embedding(data[emb_name])
                hard = refine_domains(raw, conn, mode="hard")
                spagcn = refine_domains(raw, conn, mode="spagcn")
                smoothing_examples[(unit, variant, slide, emb_name)] = dict(raw=raw, hard=hard, spagcn=spagcn)

print(f"Smoothed {len(smoothing_examples)} (unit, variant, slide, embedding) combinations.")


## 6. ARI / silhouette / NMI again, post-smoothing

Same metrics as section 3, but comparing the SMOOTHED labels (both modes)
against the same bin-level label-transfer reference -- does spatial
smoothing move the clustering closer to or further from the independent
tissue-region reference, and does it change silhouette (expected to rise
somewhat almost mechanically, since smoothing directly increases spatial
label homogeneity -- read alongside the ARI/NMI change, not instead of it).

In [ ]:
smoothed_quality_rows = []
for (unit, variant, slide, emb_name), labelsets in smoothing_examples.items():
    grid_ref = load_grid_reference(slide)
    if grid_ref is None:
        continue
    data = load_embeddings(unit, variant, slide)  # re-fetch cell_xy/z for this combo
    if data is None:
        continue
    reference = lt.assign_nearest_grid_label(data["cell_xy"], grid_ref, label_col="predicted_tissuetype_grid_image")

    for label_kind in ["raw", "hard", "spagcn"]:
        labels = labelsets[label_kind]
        z = data[emb_name]
        quality = em.cluster_quality_metrics(z, labels)
        smoothed_quality_rows.append({
            "unit": unit, "variant": variant, "held_out_slide": slide, "embedding": emb_name,
            "label_kind": label_kind,  # "raw" = pre-smoothing, for a direct before/after comparison
            "ari_vs_label_transfer": em.ari_between_labelings(labels, reference),
            "nmi_vs_label_transfer": em.nmi_between_labelings(labels, reference),
            **quality,
        })

smoothed_quality_df = pd.DataFrame(smoothed_quality_rows)
smoothed_quality_df.to_csv(basedir / "combined/03_integration/autoencoders/final_loso_clustering_quality_smoothed.csv", index=False)
smoothed_quality_df


In [ ]:
if len(smoothed_quality_df):
    # before/after pivot: one row per (unit, variant, embedding), columns = label_kind
    (smoothed_quality_df
     .groupby(["unit", "variant", "embedding", "label_kind"])[["ari_vs_label_transfer", "nmi_vs_label_transfer", "silhouette"]]
     .mean()
     .unstack("label_kind"))
else:
    print("Nothing yet.")


## 7. Gene expression prediction

Adapted from `03_integration/autoencoders/scripts/gene_expression_probe.py`
(same `ProbeMLP`/`fit_mlp_probe`/`score_predictions` logic, copied rather
than imported since that script executes its whole pipeline at import time)
-- repointed at the final run's checkpoints/RUN_TAGs and generalized over
BOTH units (the original script was nuclei-only). Fits Ridge (linear) and a
small MLP (nonlinear) probe mapping a fold's FROZEN, already-trained
`model.token_enc(...)` output -> the real (standardized) target gene matrix,
on training-slide cells, scored on the held-out slide -- see that script's
own docstring for why this needs a live model forward pass rather than
reusing the saved `z_token` directly (the contrastive alignment never forces
`z_token`/`z_trans` to be literally interchangeable, so probing from a frozen
`token_enc` output, not from `z_trans`, is the fair "can histology alone
predict expression" test).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import copy


class ProbeMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, out_dim))

    def forward(self, x):
        return self.net(x)


def fit_mlp_probe(z_train, y_train, device, val_frac=0.1, max_epochs=100, patience=8,
                   lr=1e-3, batch_size=4096, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(z_train.shape[0])
    n_val = int(len(idx) * val_frac)
    val_idx, fit_idx = idx[:n_val], idx[n_val:]

    z_fit, y_fit = torch.from_numpy(z_train[fit_idx]).to(device), torch.from_numpy(y_train[fit_idx]).to(device)
    z_val, y_val = torch.from_numpy(z_train[val_idx]).to(device), torch.from_numpy(y_train[val_idx]).to(device)

    model = ProbeMLP(z_train.shape[1], y_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    best_val, best_state, epochs_no_improve = float("inf"), None, 0
    n_fit = z_fit.shape[0]
    for _epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_fit, device=device)
        for start in range(0, n_fit, batch_size):
            batch_idx = perm[start:start + batch_size]
            opt.zero_grad()
            loss = F.mse_loss(model(z_fit[batch_idx]), y_fit[batch_idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(z_val), y_val).item()
        if val_loss < best_val - 1e-5:
            best_val, best_state, epochs_no_improve = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


def score_predictions(y_true, y_pred):
    """median_r2/median_pearson_r are the numbers to read -- mean_r2 can blow
    up to enormous negative values on a single low-variance gene, see
    gene_expression_probe.py's own score_predictions docstring."""
    r2_per_gene = r2_score(y_true, y_pred, multioutput="raw_values")
    pearson_per_gene = np.array([
        pearsonr(y_true[:, g], y_pred[:, g])[0] if np.ptp(y_pred[:, g]) > 0 else np.nan
        for g in range(y_true.shape[1])
    ])
    return {
        "mean_r2": np.nanmean(r2_per_gene), "median_r2": np.nanmedian(r2_per_gene),
        "frac_genes_r2_pos": float(np.mean(r2_per_gene > 0)),
        "mean_pearson_r": np.nanmean(pearson_per_gene), "median_pearson_r": np.nanmedian(pearson_per_gene),
    }


In [ ]:
def run_gene_expression_probe(unit, variant, held_out_slide):
    """One fold's Ridge + MLP probe, both scored. Returns a list of 2 dict
    rows (ridge, mlp), or [] if this fold's checkpoint/token data isn't
    available."""
    u = UNITS[unit]
    tag = run_tag(unit, variant)
    ckpt_path = checkpoint_dir(unit) / f"loso_token_{held_out_slide}{tag}.pt"
    if not ckpt_path.exists():
        return []

    recon = reconstruct_held_out_cells(unit, held_out_slide)
    if recon is None:
        return []
    _, _, trans_sub_held = recon

    # target matrix: whichever gene set this variant actually trained on
    # (SVG union, or the Xenium panel) -- get_gene_mask handles both, same
    # dispatcher run_loso.py/_bins.py themselves use.
    gene_panel_path = None
    if VARIANTS[variant]["gene_panel"] is not None:
        gene_panel_path = basedir / "combined/01_preprocessing/gene_panels" / f"{VARIANTS[variant]['gene_panel']}.json"
    gene_selection = "svg" if gene_panel_path is None else "svg"  # gene_panel_path overrides when set
    gene_mask = get_gene_mask(trans_sub_held, gene_selection=gene_selection, gene_panel_path=gene_panel_path)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    n_slides = ckpt["model_state_dict"]["trans_dec.net.0.weight"].shape[1] - cfg["latent_dim"]

    train_slides = [s for s in SLIDES if s != held_out_slide]
    train_token_list, train_expr_list = [], []
    for s in train_slides:
        r = reconstruct_held_out_cells(unit, s)
        if r is None:
            continue
        cell_xy_s, _, trans_sub_s = r
        token_path = basedir / s / "01_preprocessing/unimodel/metadata" / u["token_filename"]
        tok = sc.read_h5ad(token_path)
        tok_key = tok.obs["Slide_ID"].astype(str) + "_" + tok.obs["id"].astype(str)
        tok.obs = tok.obs.set_index(tok_key)
        tok = tok[tok.obs["matched"].astype(bool)].copy()
        common = trans_sub_s.obs_names.intersection(tok.obs_names)
        if len(common) == 0:
            continue
        tok_sub = tok[common]
        token_feat = np.concatenate(
            [np.asarray(tok_sub.obsm["X_token_footprint"]), np.asarray(tok_sub.obsm["X_token_halo"])], axis=1
        ).astype(np.float32)
        X = trans_sub_s[common].X[:, gene_mask]
        X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
        train_token_list.append(token_feat)
        train_expr_list.append(X.astype(np.float32))
    if not train_token_list:
        return []
    train_token = np.concatenate(train_token_list, axis=0)
    train_expr = np.concatenate(train_expr_list, axis=0)

    token_path_held = basedir / held_out_slide / "01_preprocessing/unimodel/metadata" / u["token_filename"]
    tok_held = sc.read_h5ad(token_path_held)
    tok_held_key = tok_held.obs["Slide_ID"].astype(str) + "_" + tok_held.obs["id"].astype(str)
    tok_held.obs = tok_held.obs.set_index(tok_held_key)
    tok_held = tok_held[tok_held.obs["matched"].astype(bool)].copy()
    common_held = trans_sub_held.obs_names.intersection(tok_held.obs_names)
    held_token = np.concatenate(
        [np.asarray(tok_held[common_held].obsm["X_token_footprint"]), np.asarray(tok_held[common_held].obsm["X_token_halo"])],
        axis=1,
    ).astype(np.float32)
    held_expr = trans_sub_held[common_held].X[:, gene_mask]
    held_expr = held_expr.toarray() if hasattr(held_expr, "toarray") else np.asarray(held_expr)
    held_expr = held_expr.astype(np.float32)

    token_dim = train_token.shape[1]
    model = DualAutoencoder(
        n_svg=int(gene_mask.sum()), latent_dim=cfg["latent_dim"], proj_dim=cfg["proj_dim"],
        n_slides=n_slides, encoder_type=cfg["encoder"], token_dim=token_dim,
        token_condition_on_slide=cfg.get("token_condition_on_slide", True),
        trans_condition_on_slide=cfg.get("trans_condition_on_slide", True),
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    token_scaler = StandardScaler().fit(train_token)
    train_token_std = token_scaler.transform(train_token).astype(np.float32)
    held_token_std = token_scaler.transform(held_token).astype(np.float32)
    expr_scaler = StandardScaler().fit(train_expr)
    train_expr_std = expr_scaler.transform(train_expr).astype(np.float32)
    held_expr_std = expr_scaler.transform(held_expr).astype(np.float32)

    with torch.no_grad():
        z_train = model.token_enc(torch.from_numpy(train_token_std).to(device)).cpu().numpy()
        z_held = model.token_enc(torch.from_numpy(held_token_std).to(device)).cpu().numpy()

    common_fields = {"unit": unit, "variant": variant, "held_out_slide": held_out_slide,
                      "n_train_cells": z_train.shape[0], "n_held_cells": z_held.shape[0],
                      "n_genes": held_expr_std.shape[1]}

    rows = []
    ridge = Ridge(alpha=10.0).fit(z_train, train_expr_std)
    rows.append({**common_fields, "probe": "ridge", **score_predictions(held_expr_std, ridge.predict(z_held))})

    mlp = fit_mlp_probe(z_train, train_expr_std, device)
    with torch.no_grad():
        mlp_pred = mlp(torch.from_numpy(z_held).to(device)).cpu().numpy()
    rows.append({**common_fields, "probe": "mlp", **score_predictions(held_expr_std, mlp_pred)})
    return rows


In [ ]:
# NOTE: this section does real GPU/CPU work per fold (a forward pass + Ridge
# fit + a small MLP training loop each) -- unlike sections 3-6, which only
# read already-saved arrays. Expect this cell to take a while once there are
# many finished folds; comment out unit/variant/slide combinations below if
# you only want a quick check on what's landed so far.
expr_probe_rows = []
for unit in UNITS:
    for variant in VARIANTS:
        for slide in SLIDES:
            expr_probe_rows.extend(run_gene_expression_probe(unit, variant, slide))

expr_probe_df = pd.DataFrame(expr_probe_rows)
if len(expr_probe_df):
    expr_probe_df = expr_probe_df.set_index(["unit", "variant", "held_out_slide", "probe"])
expr_probe_df.to_csv(basedir / "combined/03_integration/autoencoders/final_loso_gene_expression_probe.csv")
expr_probe_df


## 8. Cluster-type annotation via a canonical marker-gene reference

**No cell-type atlas file exists anywhere in this project** (checked: no
CellTypist model, no Azimuth/Gut-Cell-Atlas/Elmentaite reference, nothing
under `data/`) -- the only existing label vocabulary
(`functions/label_transfer.py`) is tissue-ARCHITECTURE regions (Neoplasm,
Vessel, Connective Tissue, ...), not single-cell types, which is why
sections 3/6 compare against that separately rather than reusing it here.

**What this section uses instead**: a small, hand-curated set of canonical
human colon/intestinal cell-type marker genes -- the kind of broadly-agreed
markers that recur across essentially every published colon single-cell
atlas (e.g. the Gut Cell Atlas, Smillie et al.'s UC atlas) and 10x's own
Xenium colon panel design, NOT a verbatim import of any one atlas's full
published marker table (no such file exists in this repo to import from --
fabricating one under a specific atlas's name would overstate its
provenance). **Treat this as a coarse, directionally-useful sanity check,
not a validated cell-type call** -- if precise cell-type annotation matters
for the thesis, replacing this dict with an actual downloaded reference
(e.g. a CellTypist colon model) would be a meaningfully stronger claim than
this notebook makes on its own.

Restricted to genes that actually appear in this project's own gene sets
(SVG union / Xenium panel) below, since scoring against a marker gene this
model never saw is meaningless.

In [ ]:
# Canonical colon/intestinal cell-type markers -- broadly-agreed genes that
# recur across major published colon atlases (Gut Cell Atlas/Elmentaite et al.
# 2021 Nature; Smillie et al. 2019 Cell) and 10x's own Xenium Human Colon
# panel's own design rationale, hand-curated here (see markdown above for why
# this is NOT a verbatim atlas import). Intentionally coarse (broad lineages,
# not fine subtypes) -- appropriate for sanity-checking Leiden clusters, not
# for a publication-grade cell-type call.
CANONICAL_COLON_MARKERS = {
    "Epithelial (general)": ["EPCAM", "KRT8", "KRT18", "CDH1"],
    "Colonocyte / Enterocyte": ["CA2", "CA1", "SLC26A3", "CEACAM7"],
    "Goblet": ["MUC2", "TFF3", "FCGBP", "CLCA1"],
    "Stem / TA": ["LGR5", "OLFM4", "ASCL2"],
    "Enteroendocrine": ["CHGA", "CHGB", "CPE"],
    "Tuft": ["AVIL", "AZGP1"],
    "T cell": ["CD3D", "CD3E", "CD3G", "CD2"],
    "B cell": ["CD79A", "CD79B", "BANK1", "MS4A1"],
    "Plasma": ["DERL3"],
    "Myeloid / Macrophage": ["CD14", "C1QA", "C1QB", "C1QC", "CD163"],
    "Mast": ["CPA3", "CMA1"],
    "Dendritic": ["CLEC9A"],
    "Endothelial": ["AQP1", "CEACAM1"],
    "Fibroblast / Stromal": ["COL17A1", "COL19A1", "DPYSL3"],
}

def restrict_markers_to_available_genes(marker_dict, available_genes):
    available = set(available_genes)
    restricted = {}
    for cell_type, genes in marker_dict.items():
        kept = [g for g in genes if g in available]
        if kept:
            restricted[cell_type] = kept
        else:
            print(f"  '{cell_type}': none of {genes} present in this gene set -- dropped entirely")
    return restricted


In [ ]:
def annotate_clusters_by_markers(trans_sub, cluster_labels, marker_dict):
    """
    Scores each cell type's marker set per cell (sc.tl.score_genes, standard
    scanpy approach), averages within each Leiden cluster, and assigns
    each cluster the cell type with the highest mean score -- a simple,
    transparent argmax, not a statistical test. Returns (cluster -> cell_type
    dict, per-cluster score DataFrame) for inspection before trusting the
    argmax call on any borderline cluster.
    """
    restricted = restrict_markers_to_available_genes(marker_dict, trans_sub.var_names)
    adata = trans_sub.copy()
    for cell_type, genes in restricted.items():
        sc.tl.score_genes(adata, gene_list=genes, score_name=f"score__{cell_type}")

    score_cols = [f"score__{ct}" for ct in restricted]
    scores = pd.DataFrame(adata.obs[score_cols].values, columns=list(restricted.keys()))
    scores["cluster"] = np.asarray(cluster_labels)
    per_cluster = scores.groupby("cluster").mean()
    assignment = per_cluster.idxmax(axis=1).to_dict()
    return assignment, per_cluster


annotation_rows = []
for unit in UNITS:
    for variant in VARIANTS:
        for slide in SLIDES:
            data = load_embeddings(unit, variant, slide)
            if data is None:
                continue
            leiden_labels = cluster_embedding(data["z_trans"])
            assignment, per_cluster_scores = annotate_clusters_by_markers(
                data["trans_sub"], leiden_labels, CANONICAL_COLON_MARKERS
            )
            for cluster_id, cell_type in assignment.items():
                annotation_rows.append({
                    "unit": unit, "variant": variant, "held_out_slide": slide,
                    "cluster": cluster_id, "assigned_cell_type": cell_type,
                    "n_cells_in_cluster": int((leiden_labels == cluster_id).sum()),
                })

annotation_df = pd.DataFrame(annotation_rows)
annotation_df.to_csv(basedir / "combined/03_integration/autoencoders/final_loso_cluster_annotation.csv", index=False)
annotation_df


In [ ]:
if len(annotation_df):
    print("Assigned cell-type distribution, pooled across all available (unit, variant, slide, cluster):")
    print(annotation_df["assigned_cell_type"].value_counts())
else:
    print("Nothing yet.")


## 9. Spatial cluster maps: one slide, before vs. after smoothing, across unit x gene-selection

One figure, 2 rows x 4 columns, for a single chosen slide:
- columns: nuclei+SVG, nuclei+Xenium panel, bins+SVG, bins+Xenium panel (the
  unit x gene-selection comparison specifically -- the semantic-exclusion
  variant is deliberately left out of this plot, it's a separate axis this
  figure isn't about)
- row 1: raw Leiden clustering on `z_trans` (same clustering as sections 3/6,
  the transcriptomics-only embedding -- matches thesis Section 5.6.3's own
  framing of domain identification starting from `z_trans` specifically,
  since the graph-based transcriptomics encoder already carries local-
  neighborhood information before any clustering happens)
- row 2: the SAME raw clustering, after `refine_domains` spatial smoothing
  (`SMOOTHING_MODE` below -- "hard" by default; change to "spagcn" to see
  the more conservative rule instead)

Each (unit, variant) is loaded and clustered ONCE and reused for both rows,
rather than recomputing per row.

In [ ]:
PLOT_SLIDE = "CRC_P2"      # change once more folds finish -- as of writing, most
                            # (unit, variant) combos for ANY slide aren't done yet, so
                            # don't be surprised by "no data yet" panels below
SMOOTHING_MODE = "hard"    # or "spagcn" -- see refine_domains' own docstring

PLOT_VARIANTS = ["svg", "colon_xenium_panel"]  # excludes svg_semantic_excl_raw_0.2 on purpose
PLOT_PANELS = [(unit, variant) for unit in ["nuclei", "bins"] for variant in PLOT_VARIANTS]
# -> [(nuclei,svg), (nuclei,colon_xenium_panel), (bins,svg), (bins,colon_xenium_panel)]
# i.e. exactly panels 1-4 in the requested left-to-right order


def compute_slide_clusters(unit, variant, slide, smoothing_mode=SMOOTHING_MODE):
    """Loads + clusters + smooths ONCE -- returns (cell_xy, raw_labels,
    smoothed_labels), or None if this (unit, variant, slide) isn't ready yet."""
    data = load_embeddings(unit, variant, slide)
    if data is None:
        return None
    raw = cluster_embedding(data["z_trans"])
    conn = build_slide_graph(unit, data["cell_xy"])
    smoothed = refine_domains(raw, conn, mode=smoothing_mode)
    return data["cell_xy"], raw, smoothed


def plot_cluster_panel(ax, cell_xy, labels, title):
    ax.scatter(cell_xy[:, 0], cell_xy[:, 1], c=pd.factorize(labels)[0], cmap="tab20", s=2)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{title}\n(n={len(np.unique(labels))} clusters)", fontsize=9)


fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, (unit, variant) in enumerate(PLOT_PANELS):
    result = compute_slide_clusters(unit, variant, PLOT_SLIDE)
    if result is None:
        for row in (0, 1):
            axes[row, col].text(0.5, 0.5, f"{unit} / {variant}\nno data yet",
                                 ha="center", va="center", transform=axes[row, col].transAxes)
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
        continue
    cell_xy, raw, smoothed = result
    plot_cluster_panel(axes[0, col], cell_xy, raw, f"{unit} / {variant}")
    plot_cluster_panel(axes[1, col], cell_xy, smoothed, f"{unit} / {variant} (smoothed)")

axes[0, 0].set_ylabel("before smoothing", fontsize=11)
axes[1, 0].set_ylabel(f"after smoothing ({SMOOTHING_MODE})", fontsize=11)
fig.suptitle(f"{PLOT_SLIDE}: Leiden clustering on z_trans, before vs. after spatial smoothing "
             f"({SMOOTHING_MODE} mode)", fontsize=13)
plt.tight_layout()

out_path = basedir / "combined/03_integration/autoencoders" / f"final_loso_spatial_clusters_{PLOT_SLIDE}.png"
plt.savefig(out_path, dpi=150)
print(f"Saved to {out_path}")
plt.show()
